In [2]:
!pip install -q litellm gradio requests beautifulsoup4


In [3]:
import os
from getpass import getpass

import gradio as gr
import requests

from bs4 import BeautifulSoup
from litellm import completion

e:\GENAI-PEP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
openai_api_key = getpass("Enter your OpenAI API Key: ")

In [5]:
os.environ["OPENAI_API_KEY"] = openai_api_key
MODEL = "openai/gpt-4o-mini"

In [6]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url, max_chars=2000):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)

        soup = BeautifulSoup(response.content, "html.parser")

        title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for tag in soup.body(["script", "style", "img", "input"]):
                tag.decompose()

            text = soup.body.get_text(separator="\n", strip=True)

        else:
            text = ""

        return (title + "\n\n" + text)[:max_chars]

    except Exception as e:
        return f"Error fetching website: {e}"

In [7]:
page = fetch_website_contents("https://anthropic.com")

print(page[:300])

Home \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at 


In [8]:
pamphlet_system = """
You are an expert marketing assistant.

Your task is to create a short company pamphlet in Markdown format.

Rules:
- Use ONLY the information provided from the website text.
- Do NOT make up any facts.
- Do NOT use code blocks.
- Keep the pamphlet concise and attractive.
- Use headings and bullet points.
"""

In [9]:
company_name = "Anthropic"

prompt = f"""
Company Name:
{company_name}

Website Content:

{page}

Create a short company pamphlet using only the information above.
"""

print(prompt)


Company Name:
Anthropic

Website Content:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
About Claude
Overview
Pricing
Contact sales
Models
Mythos
Fable
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Policy
Commitments
Initiatives
Claude's Constitution
Claude Corps
Policy on the AI Exponential
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers

In [10]:
response = completion(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": pamphlet_system
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

# Welcome to Anthropic

## Putting Safety at the Frontier of AI

At Anthropic, we are dedicated to harnessing the vast potential of AI while prioritizing safety and responsibility. As a public benefit corporation, our mission focuses on securing AI's benefits and mitigating its risks.

### Our Focus Areas
- **AI Research**
  - Tackling hard questions around AI safety, governance, and societal impacts.
  
### Latest Innovations
- **Opus 5**  
  A significant advancement in our Opus tier, offering:
  - Stronger coding capabilities
  - More proficient agents
  - Enhanced professional work

- **Sonnet 5**  
  Introducing our most capable Sonnet model yet.

### Join Us
Discover more about our initiatives, commitments, and how you can get involved in shaping the future of AI. 

### Learn More 
Explore our resources and join the conversation about responsible AI development! 

**Visit us at [Claude.ai](https://Claude.ai)** to try Claude and explore our offerings.


In [11]:
def stream_pamphlet(company_name, url):

    page = fetch_website_contents(url)

    prompt = f"""
Company Name:
{company_name}

Website Content:

{page}

Create a short company pamphlet using only the information above.
"""

    stream = completion(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": pamphlet_system
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream=True
    )

    result = ""

    for chunk in stream:

        if chunk.choices[0].delta.content:

            result += chunk.choices[0].delta.content

            yield result

In [12]:
demo = gr.Interface(
    fn=stream_pamphlet,
    inputs=[
        gr.Textbox(
            label="Company Name",
            placeholder="e.g. Anthropic"
        ),
        gr.Textbox(
            label="Company Website",
            placeholder="https://anthropic.com"
        )
    ],
    outputs=gr.Markdown(label="Generated Company Pamphlet"),
    title="Pamphlet Generator",
    description="Enter a company name and website URL to generate a short marketing pamphlet."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [17]:
def chat(message, history):
  response = completion(
    model=MODEL,
    messages= history + [
        {
            "role": "user",
            "content": message
        }
    ]
  )
  return response.choices[0].message.content

In [18]:
gr.ChatInterface(
    fn = chat,
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
